# Jobs: pvae metabolic cost (beta + K)

project = ```BayLearn```, host = ```any```, device = ```any```

**Motivation**: <br>

- Explore various beta
- Explore various latent dims (K)

In [1]:
# HIDE CODE


project_name = 'PoissonVAE'


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, project_name))
from figures.fighelper import *
from main.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = f"Dropbox/git/{project_name}/scripts"
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['copyfits.sh', 'fit_vae.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh']

## mach

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'

latent_dims = [32, 64, 128, 512, 768, 1024, 2048, 4096]
betas = [
    0.01, 0.1, 0.5, 1,
    1.5, 2, 4, 8,
]

len(betas), len(latent_dims)

(8, 8)

In [6]:
for kl_beta in betas:
    for k in latent_dims:
        # get arg
        arg = ' '.join([
            f"--n_latents {k}",
            f"--kl_beta {kl_beta}",
            '--comment MetabolicCost',
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset='vH16',
            model=model_type,
            archi='lin|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

64

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 16, 1: 16, 2: 16, 3: 16}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 0.01 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 0.01 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 0.1 --comment MetabolicCost --verbose
&& 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 0.1 --comment MetabolicCost 
--verbose

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 0.5 --comment MetabolicCost --verbose
&& 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 0.5 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 1 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '0' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 1 --comment MetabolicCost --verbose

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


...

./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 0.5 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 0.5 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 1 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 1 --comment MetabolicCost --verbose

[PROGRESS] 'mach-cuda3-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 1.5 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 1.5 --comment MetabolicCost 
--verbose && 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 2 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 2 --comment MetabolicCost --verbose

[PROGRESS] 'mach-cuda3-fit3.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 4 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 4 --comment MetabolicCost --verbose
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 8 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 8 --comment MetabolicCost --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 4 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 4 --comment MetabolicCost --verbose
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 8 --comment MetabolicCost --verbose 
&& 
./fit_vae.sh '3' 'vH16' 'poisson' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 8 --comment MetabolicCost --verbose

## mach (only missing)

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'poisson'

missing_combos = [
    (2048, 0.01),
    (2048, 0.1),
    (4096, 0.01),
    (4096, 0.1),
    (4096, 0.5),
    (4096, 1),
    (4096, 4),
    (4096, 8),
]
len(missing_combos)

8

In [6]:
for k, kl_beta in missing_combos:
    # get arg
    arg = ' '.join([
        f"--n_latents {k}",
        f"--kl_beta {kl_beta}",
        '--comment MetabolicCost',
        '--verbose',
        # '--dry_run',
    ])
    gpu_i = tot % torch.cuda.device_count()
    scripts[gpu_i].append(job_runner_script(
        device=gpu_i,
        dataset='vH16',
        model=model_type,
        archi='lin|lin',
        args=arg,
        seed=0,
    ))
    tot += 1

In [7]:
print(tot)

8

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 2, 1: 2, 2: 2, 3: 2}

### Save

In [9]:
n_fits = 2

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        # print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda1-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda2-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda2-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda3-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts
[PROGRESS] 'mach-cuda3-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


## chewie

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_type = 'gaussian'
latent_act = 'relu'

latent_dims = [32, 64, 128, 512, 768, 1024, 2048, 4096]
betas = [
    0.01, 0.1, 0.5, 1,
    1.5, 2, 4, 8,
]

len(betas), len(latent_dims)

(8, 8)

In [6]:
for kl_beta in betas:
    for k in latent_dims:
        # get arg
        arg = ' '.join([
            f"--n_latents {k}",
            f"--kl_beta {kl_beta}",
            f"--latent_act {latent_act}",
            '--comment MetabolicCost',
            '--verbose',
            # '--dry_run',
        ])
        gpu_i = tot % torch.cuda.device_count()
        scripts[gpu_i].append(job_runner_script(
            device=gpu_i,
            dataset='vH16',
            model=model_type,
            archi='lin|lin',
            args=arg,
            seed=0,
        ))
        tot += 1

In [7]:
print(tot)

64

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 32, 1: 32}

### Save

In [9]:
n_fits = 8

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'chewie-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 0.01 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 128 --kl_beta 0.01 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 0.01 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 2048 --kl_beta 0.01 --latent_act relu --comment 
MetabolicCost --verbose

[PROGRESS] 'chewie-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 32 --kl_beta 0.1 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 128 --kl_beta 0.1 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 768 --kl_beta 0.1 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 2048 --kl_beta 0.1 --latent_act relu --comment 
MetabolicCost --verbose

[PROGRESS] 'chewie-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


...

./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 64 --kl_beta 2 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 2 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 1024 --kl_beta 2 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 2 --latent_act relu --comment 
MetabolicCost --verbose

[PROGRESS] 'chewie-cuda1-fit6.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 64 --kl_beta 4 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 4 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 1024 --kl_beta 4 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 4 --latent_act relu --comment 
MetabolicCost --verbose

[PROGRESS] 'chewie-cuda1-fit7.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 64 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 1024 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 64 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 512 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 1024 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'lin|lin' --seed 0 --n_latents 4096 --kl_beta 8 --latent_act relu --comment 
MetabolicCost --verbose

## mach:  $K \in \{192, 256, 384\}$

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
model_combos = [
    ('poisson', None),
    ('gaussian', 'relu'),
]
latent_dims = [192, 256, 384]
betas = [
    0.01, 0.1, 0.5, 1,
    1.5, 2, 4, 8,
]

len(betas), len(latent_dims), len(model_combos)

(8, 3, 2)

In [6]:
for model_type, latent_act in model_combos:
    for kl_beta in betas:
        for k in latent_dims:
            # get arg
            arg = ' '.join([
                f"--n_latents {k}",
                f"--kl_beta {kl_beta}",
                f"--latent_act {latent_act}",
                '--comment MetabolicCost',
                '--verbose',
                # '--dry_run',
            ])
            gpu_i = tot % torch.cuda.device_count()
            scripts[gpu_i].append(job_runner_script(
                device=gpu_i,
                dataset='vH16',
                model=model_type,
                archi='lin|lin',
                args=arg,
                seed=0,
            ))
            tot += 1

In [7]:
print(tot)

48

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 12, 1: 12, 2: 12, 3: 12}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            verbose=False,
            mode='txt',
        )